In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from arch import arch_model
import datetime as dt
from tensorflow import keras 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from numpy.random import seed
#from tensorflow import set_random_seed
import tensorflow
from keras.models import load_model

In [2]:
stocks = ['China','Hongkong','Indonesia','Japan','Korea','Singapore']
full_data = {}

for stock in stocks:
    full_data[stock] = pd.read_csv(f'/Users/khai/Desktop/Thesis_Asia_v2/DataSet/{stock}.csv',parse_dates=['Date'],
                                        index_col=['Date'])

In [3]:
start = dt.datetime(2011,2,1)
end = dt.datetime(2021,12,31)
split_date = dt.datetime(2020,12,31)
#ret = df['Close']

def implement_algo(stock):
    ctg = full_data[stock]
    df = np.log(ctg[['Close']]).diff().dropna()*100
    df['std'] = df['Close'].rolling(window=2, center=False).std()
    df['std'] = df['std']/np.sqrt(2)
    df = df.dropna()
    df['target_vol'] = df['std'].shift(-1)
    df = df.dropna()
    
    df['ret_previous'] = df['Close'].shift(1)
    #df['square_log_previous'] = np.log(df['ret_previous']+1)**2
    df['std_previous'] = df['std'].shift(1)
    df = df.dropna()
    
    df_ind = ctg.copy()
    df_ind['spread'] = (df_ind['Open']-df_ind['Close'])/(df_ind['High']-df_ind['Low'])
    df_ind['Volume_diff'] = df_ind['Volume'].pct_change()
    df_ind_final = df_ind.loc[df.index,['spread','Volume_diff']]
    df = pd.concat([df,df_ind_final],axis=1)
    
    #### Fit the garch model #####
    
    df_train = df['2011-02-01':'2020-12-31']
    df_test = df['2021-01-01':]
    ret = df['Close']
    #am = arch_model(ret,mean='AR',vol='Garch', p=1, o=0, q=1, dist='skewt')
    am = arch_model(ret,mean='AR',vol='EGARCH', p=1, o=0, q=1)
    res = am.fit(last_obs=split_date,disp='off')
    res.summary()
    
    #### Use garch model to predict mean,vol ####
    forecasts_train = res.forecast(horizon=1, start=start)
    variance_pred = np.sqrt(forecasts_train.variance[start:].shift(1).dropna())
    mean_pred = forecasts_train.mean[start:]
    
    df_train['forecast_vol_garch'] = variance_pred[start:split_date]
    df_train['garch_mean'] = mean_pred[start:split_date]

    df_test['forecast_vol_garch'] = variance_pred[split_date:]
    df_test['garch_mean'] = mean_pred[split_date:]
    
    df_train = df_train.dropna() 
    df_train.replace([np.inf, -np.inf], 0, inplace=True)
    df_test.replace([np.inf, -np.inf], 0, inplace=True)
    
        #df_test = df_test.dropna()
    
    #### Build ANN Model ####
    
#     df_train['square_log_ret'] = np.log(df_train['Close']+1)**2
#     df_test['square_log_ret'] = np.log(df_test['Close']+1)**2
    
    X_train = df_train[['ret_previous','std_previous','forecast_vol_garch','spread','Volume_diff']].values
    y_train = df_train[['target_vol']].values

    X_test = df_test[['ret_previous','std_previous','forecast_vol_garch','spread','Volume_diff']].values
    y_test = df_test[['target_vol']].values
    
    model = load_model(f'/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/ANN_Model/my_model_ANN_{stock}.h5')

    prediction_train = model.predict(X_train)

    #### Standardized residual calculation ####
    df_train['ANN_vol'] = prediction_train
    df_train['standard_residual'] = (df_train['Close']-df_train['garch_mean'])/df_train['ANN_vol']
    df_train['GARCH_standard_residual'] = (df_train['Close']-df_train['garch_mean'])/df_train['forecast_vol_garch']
    std_res = df_train['standard_residual']
    garch_std_res = df_train['GARCH_standard_residual']

    #### Prediction on test set ####
    prediction_test = model.predict(X_test)
    df_test['ANN_vol'] = prediction_test
    df_test['standard_residual'] = (df_test['Close']-df_test['garch_mean'])/df_test['ANN_vol']
    df_test['GARCH_standard_residual'] = (df_test['Close']-df_test['garch_mean'])/df_test['forecast_vol_garch']


    return df_train,df_test

In [4]:
import sklearn.metrics as metrics
def calculate_metrics(y,yhat):
    mae = metrics.mean_absolute_error(y, yhat)
    mse = metrics.mean_squared_error(y, yhat)
    rmse = np.sqrt(mse) # or mse**(0.5)  
    r2 = metrics.r2_score(y,yhat)
    print("Results of sklearn.metrics:")
    print("MAE:",mae)
    print("MSE:", mse)
    print("RMSE:", rmse)
    print("R-Squared:", r2)

In [5]:
def evaluate_vol_prediction(df_train,df_test):
    
    y = list(df_train['target_vol'])
    yhat_ANN = list(df_train['ANN_vol'])
    yhat_Garch = list(df_train['forecast_vol_garch'])

    print('ANN Prediction Train')
    calculate_metrics(y,yhat_ANN)
    print('\n')

    print('Garch Prediction Train')
    calculate_metrics(y,yhat_Garch)
    
    y = list(df_test['target_vol'])
    yhat_ANN = list(df_test['ANN_vol'])
    yhat_Garch = list(df_test['forecast_vol_garch'])

    print('ANN Prediction Test')
    calculate_metrics(y,yhat_ANN)
    print('\n')

    print('Garch Prediction Test')
    calculate_metrics(y,yhat_Garch)
    
    df_train[['forecast_vol_garch','target_vol']].plot()
    df_train[['ANN_vol','target_vol']].plot()
    plt.title('Train')
    plt.plot()
    
    df_test[['forecast_vol_garch','target_vol']].plot()
    df_test[['ANN_vol','target_vol']].plot()
    plt.title('Test')
    plt.plot()

In [ ]:
train_data_dict = {}
test_data_dict = {}
for stock in stocks:
    df_train,df_test = implement_algo(stock)
    train_data_dict[stock] = df_train
    test_data_dict[stock] = df_test
    df_train.to_csv(f'/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Train_Test_Residual/df_train_{stock}.csv')
    df_test.to_csv(f'/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Train_Test_Residual/df_test_{stock}.csv')
    print(stock)





In [8]:
df = train_data_dict['China']['standard_residual']
for stock in stocks[1:]:
    df_new = train_data_dict[stock]['standard_residual']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_std_train.csv')
print('Done')

Done


In [10]:
df = train_data_dict['China']['GARCH_standard_residual']
for stock in stocks[1:]:
    df_new = train_data_dict[stock]['GARCH_standard_residual']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/Garch_std_train.csv')
print('Done')

Done


IMPORT VOL GARCH ANN

In [11]:
df = train_data_dict['China']['forecast_vol_garch']
for stock in stocks[1:]:
    df_new = train_data_dict[stock]['forecast_vol_garch']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.head()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/garch_vol_train.csv')
print('Done')

Done


In [12]:
df = train_data_dict['China']['garch_mean']
for stock in stocks[1:]:
    df_new = train_data_dict[stock]['garch_mean']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/mean_train.csv')
print('Done')
df.head()

Done


,China,Hongkong,Indonesia,Japan,Korea,Singapore
Date,,,,,,
2011-02-09,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175
2011-02-10,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175
2011-02-14,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175
2011-02-16,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175
2011-02-17,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175


In [13]:
df = train_data_dict['China']['ANN_vol']
for stock in stocks[1:]:
    df_new = train_data_dict[stock]['ANN_vol']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_vol_train.csv')
print('Done')
df.head()

Done


,China,Hongkong,Indonesia,Japan,Korea,Singapore
Date,,,,,,
2011-02-09,0.621450,0.599347,0.888289,0.578744,0.486799,0.389664
2011-02-10,0.814915,0.625752,0.819560,0.433352,0.634647,0.659943
2011-02-14,0.959714,0.523540,0.380280,0.435508,0.629078,0.413146
2011-02-16,0.616156,0.440427,0.443476,0.489025,0.499117,0.364643
2011-02-17,0.520743,0.510590,0.404179,0.442974,0.648953,0.420029


IMPORT TEST DATA

In [14]:
df = test_data_dict['China']['standard_residual']
for stock in stocks[1:]:
    df_new = test_data_dict[stock]['standard_residual']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_std_test.csv')
print('Done')
df.head()

Done


,China,Hongkong,Indonesia,Japan,Korea,Singapore
Date,,,,,,
2021-01-04,2.278405,1.275099,4.384264,-1.031964,3.703886,1.638453
2021-01-05,1.336621,1.201201,1.216607,-0.801895,1.273191,0.027761
2021-01-06,1.897124,0.318894,-2.565076,-0.807244,-1.982129,0.330499
2021-01-07,1.635367,-1.077203,2.072673,2.962045,4.005473,5.360888
2021-01-08,-0.588174,2.158839,3.103282,4.822489,5.312446,6.512496


In [15]:
df = test_data_dict['China']['GARCH_standard_residual']
for stock in stocks[1:]:
    df_new = test_data_dict[stock]['GARCH_standard_residual']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/Garch_std_test.csv')
print('Done')
df.head()

Done


,China,Hongkong,Indonesia,Japan,Korea,Singapore
Date,,,,,,
2021-01-04,0.802284,0.844360,1.695726,-0.690875,2.233878,0.728674
2021-01-05,0.677056,0.601232,0.357476,-0.404119,1.242471,0.014406
2021-01-06,0.579520,0.121907,-0.980857,-0.433758,-0.605781,0.152842
2021-01-07,0.666413,-0.565593,1.083799,1.522968,1.670884,2.466733
2021-01-08,-0.183388,1.201518,1.233630,2.012945,2.868619,3.948953


In [16]:
df = test_data_dict['China']['forecast_vol_garch']
for stock in stocks[1:]:
    df_new = test_data_dict[stock]['forecast_vol_garch']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/garch_vol_test.csv')
print('Done')
df.head()

Done


,China,Hongkong,Indonesia,Japan,Korea,Singapore
Date,,,,,,
2021-01-04,1.044301,1.010706,1.195521,1.085628,1.081609,0.702712
2021-01-05,1.052266,1.018219,1.328043,1.082291,1.231591,0.701410
2021-01-06,1.049275,1.010444,1.253793,1.036509,1.278484,0.649060
2021-01-07,1.037910,0.973775,1.277225,0.998899,1.251911,0.610769
2021-01-08,1.034198,0.965028,1.316328,1.123870,1.350389,0.735807


In [17]:
df = test_data_dict['China']['garch_mean']
for stock in stocks[1:]:
    df_new = test_data_dict[stock]['garch_mean']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/mean_test.csv')
print('Done')
df.head()

Done


,China,Hongkong,Indonesia,Japan,Korea,Singapore
Date,,,,,,
2021-01-04,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175
2021-01-05,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175
2021-01-06,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175
2021-01-07,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175
2021-01-08,0.019083,0.030198,0.055305,0.07076,0.023985,0.017175


In [18]:
df = test_data_dict['China']['ANN_vol']
for stock in stocks[1:]:
    df_new = test_data_dict[stock]['ANN_vol']
    df = pd.concat([df,df_new],axis=1)
df = df.dropna()
df.columns = stocks
df = df.dropna()
df.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_vol_test.csv')
print('Done')
df.head()

Done


,China,Hongkong,Indonesia,Japan,Korea,Singapore
Date,,,,,,
2021-01-04,0.367725,0.669281,0.462398,0.726802,0.652337,0.312519
2021-01-05,0.533018,0.509645,0.390219,0.545426,1.201874,0.363982
2021-01-06,0.320525,0.386275,0.479437,0.556949,0.390732,0.300164
2021-01-07,0.422949,0.511288,0.667860,0.513595,0.522235,0.281036
2021-01-08,0.322454,0.537094,0.523272,0.469112,0.729184,0.446168


COMBINE BOTH TRAIN/TEST

In [19]:
ANN_std_train = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_std_train.csv')
ANN_std_test = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_std_test.csv')

ANN_std_final = pd.concat([ANN_std_train,ANN_std_test],axis=0)
ANN_std_final.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_std_full.csv')

In [20]:
garch_std_train = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/Garch_std_train.csv')
garch_std_test = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/Garch_std_test.csv')

garch_std_final = pd.concat([garch_std_train,garch_std_test],axis=0)
garch_std_final.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/Garch_std_full.csv')

In [21]:
ANN_vol_train = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_vol_train.csv')
ANN_vol_test = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_vol_test.csv')

ANN_vol_final = pd.concat([ANN_vol_train,ANN_vol_test],axis=0)
ANN_vol_final.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/ANN_vol_full.csv')

In [22]:
garch_vol_train = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/garch_vol_train.csv')
garch_vol_test = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/garch_vol_test.csv')

garch_vol_final = pd.concat([garch_vol_train,garch_vol_test],axis=0)
garch_vol_final.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/Garch_vol_full.csv')

In [23]:
mean_train = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/mean_train.csv')
mean_test = pd.read_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/mean_test.csv')

mean_final = pd.concat([mean_train,mean_test],axis=0)
mean_final.to_csv('/Users/khai/Desktop/Thesis_Asia_v2/CodingFiles/Residual/mean_full.csv')


In [24]:
garch_vol_final

,Date,China,Hongkong,Indonesia,Japan,Korea,Singapore
0,2011-02-09,1.399660,0.958022,1.306276,1.052824,0.853693,0.717229
1,2011-02-10,1.388605,1.003639,1.329976,0.985160,0.906370,0.781096
2,2011-02-14,1.370106,1.067208,1.279914,0.919495,1.093338,0.901474
3,2011-02-16,1.407536,1.105933,1.234993,0.911631,1.119708,0.919959
4,2011-02-17,1.388984,1.121768,1.125992,0.898544,1.137255,0.887579
...,...,...,...,...,...,...,...
210,2021-12-30,0.828577,1.111032,0.545474,1.225655,0.896609,0.635372
211,2022-01-04,0.829694,1.075291,0.663859,1.163361,0.885358,0.582273
212,2022-01-05,0.801925,1.029379,0.659511,1.285773,0.858882,0.706414
213,2022-01-06,0.844772,1.088496,0.674322,1.157872,0.912468,0.711427
